<a href="https://colab.research.google.com/github/Pramuuu/AI-Voice-Assistance/blob/main/AI_VOICE_ASSISTANCE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# Required Libraries
!pip install webrtcvad pydub faster-whisper transformers


In [ ]:
# Importing Libraries
import wave
import webrtcvad
from pydub import AudioSegment
from faster_whisper import WhisperModel
import os

In [ ]:
# Convert audio file (M4A or MP3) to WAV and ensure correct format
def convert_audio_to_wav(input_file, wav_file):
    file_extension = os.path.splitext(input_file)[1].lower()

    if file_extension in [".m4a", ".mp3"]:
        audio = AudioSegment.from_file(input_file, format=file_extension[1:])
        audio = audio.set_frame_rate(32000).set_channels(1)    # Sample rate 32KHz (determines how many audio samples are taken per second)
        audio.export(wav_file, format="wav")
    else:
        raise ValueError("Unsupported file format. Please provide a .m4a or .mp3 file.")


In [ ]:
# Read WAV file
def read_wave(path):
    with wave.open(path, 'rb') as wf:
        pcm_data = wf.readframes(wf.getnframes())
        sample_rate = wf.getframerate()
    return pcm_data, sample_rate

# Frame generator for VAD
def frame_generator(frame_duration_ms, audio, sample_rate):
    n = int(sample_rate * (frame_duration_ms / 1000.0) * 2)
    offset = 0
    while offset + n <= len(audio):
        yield audio[offset:offset + n]
        offset += n

In [ ]:
# Apply VAD filtering
def vad_filter(audio_path, vad_mode=1, vad_threshold=0.5):
    vad = webrtcvad.Vad(vad_mode)
    pcm_data, sample_rate = read_wave(audio_path)

    frames = frame_generator(30, pcm_data, sample_rate)           # Frame duration 30ms
    frames_with_voice = [frame for frame in frames if vad.is_speech(frame, sample_rate)]

    filtered_audio = b''.join(frames_with_voice)
    return filtered_audio, sample_rate

In [ ]:
# Save filtered audio to WAV
def save_wave(file_name, pcm_data, sample_rate):
    with wave.open(file_name, 'wb') as wf:
        wf.setnchannels(1)          # Audio Channel:1 (mono)
        wf.setsampwidth(2)          # Sample width 2 bytes (16bit)
        wf.setframerate(sample_rate)
        wf.writeframes(pcm_data)

In [ ]:
# Transcribe audio using faster-whisper
def transcribe_audio(input_file):
    # Convert to WAV
    wav_path = "temp_audio.wav"
    convert_audio_to_wav(input_file, wav_path)

    # Apply VAD filter
    filtered_audio, sample_rate = vad_filter(wav_path, vad_mode=1, vad_threshold=0.5)        # VAD Threshold 0.5

    # Save filtered audio
    filtered_audio_path = "filtered_audio.wav"
    save_wave(filtered_audio_path, filtered_audio, sample_rate)

    # Load Whisper model
    model = WhisperModel("base")

    # Transcribe the filtered audio
    segments, info = model.transcribe(filtered_audio_path, language="en", beam_size=5)

    # Collect transcription
    transcribed_text = " ".join(segment.text for segment in segments)

    return transcribed_text

In [ ]:
# Transcribing Text
audio_path = "/content/Recording (3).m4a"  # Replace with your audio file (.m4a or .mp3)
transcribed_text = transcribe_audio(audio_path)
print("Transcribed Text:", transcribed_text)


Transcribed Text:  What is machine learning?


In [ ]:
pip install opencv-python


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load the pre-trained LLaMA model and tokenizer
model_name = "openlm-research/open_llama_3b"  # Model name
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Move model to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 3200, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=3200, out_features=3200, bias=False)
          (k_proj): Linear(in_features=3200, out_features=3200, bias=False)
          (v_proj): Linear(in_features=3200, out_features=3200, bias=False)
          (o_proj): Linear(in_features=3200, out_features=3200, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=3200, out_features=8640, bias=False)
          (up_proj): Linear(in_features=3200, out_features=8640, bias=False)
          (down_proj): Linear(in_features=8640, out_features=3200, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((3200,), eps=1e-06)
        (post_attention_layernorm): LlamaRMSNorm((3200,), eps=1e-06)
      )
    )
    (norm): LlamaRMSNorm((3200,), eps=1e-06)
 

In [ ]:

# Tokenize the input text
inputs = tokenizer(transcribed_text, return_tensors="pt").to(device)

# Generate a concise response from the LLaMA model
outputs = model.generate(
    inputs.input_ids,
    num_return_sequences=1,     # Ensure only one response is generated
    do_sample=True,             # Enable sampling to use top_k, top_p, and temperature
    top_k=50,                   # Consider only the top 50 tokens to reduce randomness
    top_p=0.9,                  # Nucleus sampling with top_p (cumulative probability)
    temperature=0.7,            # Lower temperature for less randomness
    no_repeat_ngram_size=3,     # Prevent repeating any 3-word sequences
    max_new_tokens=200             # Limit the number of new tokens generated
)

# Decode the response
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(response)


 What is machine learning?
What is Machine Learning?
Machine Learning is a branch of computer science that uses algorithms and statistical models to solve complex problems and make predictions. It is a sub-field of artificial intelligence that aims to create systems that can learn and improve themselves without being explicitly programmed.
Machines can learn from data, and can make predictions based on the data. Machine learning is a way to create machines that can do this.
What are the applications of machine learning in healthcare?
The applications of Machine Learning in healthcare include:
1. Predictive analytics: Predictive Analytics is a form of Machine learning that uses data to predict future events. It can be used to predict the risk of disease, the success of a treatment, or the likelihood of an event occurring.
2. Risk stratification: Risk stratifications are used to identify patients at risk for a particular disease. Machine Learning can be applied to this task to create a r

In [ ]:
# Remove the transcribed text from the beginning of the response
if response.startswith(transcribed_text):
    response = response[len(transcribed_text):].strip()

print("LLM Response:", response)

LLM Response: What is Machine Learning?
Machine Learning is a branch of computer science that uses algorithms and statistical models to solve complex problems and make predictions. It is a sub-field of artificial intelligence that aims to create systems that can learn and improve themselves without being explicitly programmed.
Machines can learn from data, and can make predictions based on the data. Machine learning is a way to create machines that can do this.
What are the applications of machine learning in healthcare?
The applications of Machine Learning in healthcare include:
1. Predictive analytics: Predictive Analytics is a form of Machine learning that uses data to predict future events. It can be used to predict the risk of disease, the success of a treatment, or the likelihood of an event occurring.
2. Risk stratification: Risk stratifications are used to identify patients at risk for a particular disease. Machine Learning can be applied to this task to create a risk profile f

Step 3 -> Text To Speech Conversion

In [ ]:
pip install torchaudio soundfile numpy


In [ ]:
pip install --upgrade transformers


In [ ]:
pip install accelerate sentencepiece opencv-python


In [ ]:
pip install --upgrade transformers torch sentencepiece numpy soundfile


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 105.6 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
parler-tts 0.2.2 requires transformers<=4.46.1,>=4.46.1, but you have transformers 4.49.0 which is incompatible.
numba 0.61.0 requires numpy<2.2,>=1.24, but you have numpy 2.2.4 which is incompatible.


In [ ]:
pip install git+https://github.com/huggingface/transformers

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-pimze0d1
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-pimze0d1
  Resolved https://github.com/huggingface/transformers to commit f19d018bfff1613ba05dcbf7e82c461d49aee73e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
pip install --upgrade torch torchaudio


In [ ]:
pip install datasets


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.2/69.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.4/487.4 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.9/274.9 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.3/231.3 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.1/344.1 kB 23.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0


In [ ]:
import torch
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
from datasets import load_dataset
import numpy as np
import soundfile as sf

# Function to limit the number of sentences
def limit_sentences(text, max_sentences=2):
    sentences = text.split('.')
    limited_text = '.'.join(sentences[:max_sentences]) + '.'
    return limited_text

# Function to apply Voice Activity Detection (VAD)
def apply_vad(audio_arr, vad_threshold=0.01):
    # Thresholding audio to remove low-energy segments
    audio_arr = np.where(np.abs(audio_arr) > vad_threshold, audio_arr, 0)
    return audio_arr

# Function to convert text to speech with tunable parameters
def text_to_speech_speecht5(prompt, description, output_file, pitch=1.0, gender='female', speed=1.0, vad_threshold=0.01):
    # Check device availability
    device = "cuda:0" if torch.cuda.is_available() else "cpu"

    # Limit the description to a maximum of 2 sentences
    description = limit_sentences(description)

    # Adjust the description based on the gender
    if gender == 'male':
        description = description.replace("female", "male")
    else:
        description = description.replace("male", "female")

    # Load the SpeechT5 model, processor, and vocoder
    processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
    model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts").to(device)
    vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan").to(device)

    # Load speaker embeddings
    speaker_embeddings_dataset = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")
    speaker_embeddings = torch.tensor(speaker_embeddings_dataset[0]["xvector"]).unsqueeze(0).to(device)

    # Encode the description and prompt
    inputs = processor(text=description, return_tensors="pt").to(device)

    # Generate speech
    with torch.no_grad():
        speech = model.generate_speech(inputs["input_ids"], speaker_embeddings=speaker_embeddings, vocoder=vocoder)

    # Convert speech to numpy array
    audio_arr = speech.cpu().numpy().squeeze()

    # Apply Voice Activity Detection (VAD)
    audio_arr = apply_vad(audio_arr, vad_threshold=vad_threshold)

     # Save the audio file with the correct sampling rate (16 kHz)
    sf.write(output_file, audio_arr, 16000)

    # Save the audio file
    # sf.write(output_file, audio_arr, model.config.sampling_rate)
    # print(f"Audio saved to {output_file}")
    print(f"Audio saved to {output_file}")


In [ ]:
text_prompt = response
voice_description = response
output_file = "output2.wav"

# Adjust tunable parameters here
text_to_speech_speecht5(text_prompt, voice_description, output_file, pitch=1.0, gender='male', speed=1.0, vad_threshold=0.01)



Audio saved to output2.wav
